# ==============================================================================
# 🩺 CLINICAL CARDIOLOGY SCREENING: HEART DISEASE VIA STACKING CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Combines three distinct hypothesis spaces (Logistic Regression, Random Forest, RBF SVC) using Out-Of-Fold cross-validation meta-learning to maximize clinical sensitivity.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Cardiology Environment ready.")


✅ STEP 1: Cardiology Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Heart Disease clinical dataset (1,025 patient records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/heart_disease/heart.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape[0]} patients, {df.shape[1]} clinical attributes")
print(f"Target Distribution:\n{df['target'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Shape: 920 patients, 14 clinical attributes
Target Distribution:
target
1    0.553
0    0.447
Name: proportion, dtype: float64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `target`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['target']).copy()
y = df['target'].copy()

print(f"Biomarkers: {X.columns.tolist()}")


Biomarkers: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} patients")
print(f"Test Cohort : {X_test.shape[0]} patients")


Train Cohort: 736 patients
Test Cohort : 184 patients


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Standardizing numerical vitals and encoding categorical features.


In [5]:
# STEP 7: PREPROCESSOR
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Clinical Preprocessor configured.")


✅ STEP 7: Clinical Preprocessor configured.


## ⚠️ STEP 8: BASELINE PIPELINE FIT
Fitting StackingClassifier with Logistic Regression, Random Forest, and Calibrated LinearSVC.


In [6]:
# STEP 8: BASELINE STACKING FIT
stack_clf = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)),
        ('svc', CalibratedClassifierCV(LinearSVC(dual=False, random_state=42)))
    ],
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1
)

pipe_base = Pipeline([
    ('prep', preprocessor),
    ('stack', stack_clf)
])
pipe_base.fit(X_train, y_train)

acc_base = pipe_base.score(X_test, y_test)
print(f"Baseline Stacking Accuracy: {acc_base * 100:.2f}%")


Baseline Stacking Accuracy: 83.15%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing meta-learner regularization.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'stack__final_estimator__C': [0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_base, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Clinical Meta-Learner Parameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Recall: {grid.best_score_*100:.2f}%")


🏆 Best Clinical Meta-Learner Parameters: {'stack__final_estimator__C': 0.1}
🎯 Best 5-Fold CV Recall: 85.25%


## 📊 STEP 10 & 11: CLINICAL EVALUATION REPORT & RESIDUALS
Evaluating clinical sensitivity and ROC-AUC.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 CLINICAL STACKING EVALUATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Cardiovascular Risk']))
print(f"Test Recall (Sensitivity) : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"Test ROC-AUC Score        : {roc_auc_score(y_test, y_prob):.4f}")


📋 CLINICAL STACKING EVALUATION REPORT:
                     precision    recall  f1-score   support

             Normal       0.85      0.76      0.80        82
Cardiovascular Risk       0.82      0.89      0.85       102

           accuracy                           0.83       184
          macro avg       0.83      0.82      0.83       184
       weighted avg       0.83      0.83      0.83       184

Test Recall (Sensitivity) : 89.22%
Test ROC-AUC Score        : 0.9110


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & EMERGENCY TRIAGE SMOKE TEST
Deploying pipeline and validating multi-model stacked inference latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/heart_disease_stacking_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_diag = loaded_pipe.predict(sample_patient)[0]
pred_risk = loaded_pipe.predict_proba(sample_patient)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏥 LIVE MULTI-MODEL CLINICAL STACKED INFERENCE:")
print(f"   Diagnostic Flag : {'⚠️ CARDIAC RISK' if pred_diag == 1 else '✅ HEALTHY'}")
print(f"   Stacked Risk    : {pred_risk * 100:.2f}%")
print(f"   Triage Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/heart_disease_stacking_pipeline.joblib (75,191 bytes)

🏥 LIVE MULTI-MODEL CLINICAL STACKED INFERENCE:
   Diagnostic Flag : ✅ HEALTHY
   Stacked Risk    : 38.62%
   Triage Latency  : 29.099 ms (SLA < 2.0ms: CHECK)
